In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

DATA = Path("data/processed")

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw" / "m5"

print(sys.executable)
print(RAW, RAW.exists())

In [ ]:
calendar = pd.read_csv(RAW / "calendar.csv")
calendar.head()

In [ ]:
print(calendar.shape)
print(calendar["date"].min(), calendar["date"].max())
calendar["event_name_1"].value_counts().head(10)

In [ ]:
sales_peek = pd.read_csv(RAW / "sales_train_evaluation.csv", nrows=5)
print(sales_peek.shape)
sales_peek.iloc[:, :10]

In [ ]:
ids = pd.read_csv(
    RAW / "sales_train_evaluation.csv",
    usecols=["item_id", "cat_id", "store_id", "state_id"],
)
print(ids.shape)
print(ids["store_id"].nunique(), "stores")
print(ids["item_id"].nunique(), "items")
ids["store_id"].value_counts()

In [ ]:
ca1 = ids[ids["store_id"] == "CA_1"]
ca1["cat_id"].value_counts()

In [ ]:
items = ids[["item_id", "cat_id"]].drop_duplicates()
sample = items.groupby("cat_id").sample(n=50, random_state=42)

print(len(items), "unique items in total")
print(sample["cat_id"].value_counts())
sample.head()

In [ ]:
STORES = ["CA_1", "TX_1", "WI_1"]

day_cols = {f"d_{i}": "int16" for i in range(1, 1942)}
sales = pd.read_csv(RAW / "sales_train_evaluation.csv", dtype=day_cols)
print("full file:", sales.shape)

keep = sales["store_id"].isin(STORES) & sales["item_id"].isin(sample["item_id"])
sales = sales[keep]
print("after filtering:", sales.shape)

In [ ]:
d_cols = [c for c in sales.columns if c.startswith("d_")]

totals = sales[d_cols].astype("int64").sum(axis=1)

print((totals == 0).sum(), "series with zero sales ever")
totals.describe()

In [ ]:
id_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

long = sales.melt(
    id_vars=id_cols,
    value_vars=d_cols,
    var_name="d",
    value_name="units",
)
print(long.shape)
long.head()

In [ ]:
calendar["date"] = pd.to_datetime(calendar["date"])

long = long.merge(calendar[["d", "date", "wm_yr_wk"]], on="d", how="left")
print(long.shape)
long.head()

In [ ]:
prices = pd.read_csv(RAW / "sell_prices.csv")
print("full prices file:", prices.shape)

prices = prices[prices["store_id"].isin(STORES) & prices["item_id"].isin(sample["item_id"])]
print("after filtering:", prices.shape)

keys = ["store_id", "item_id", "wm_yr_wk"]
print(prices.duplicated(subset=keys).sum(), "duplicate keys")

long = long.merge(prices, on=keys, how="left")
print(long.shape)
print(f"missing price: {long['sell_price'].isna().mean():.1%}")
long.head()

In [ ]:
no_price = long["sell_price"].isna()

print(long.loc[no_price, "units"].sum(), "units sold on rows with no price")
print(no_price.sum(), "rows with no price")

In [ ]:
long = long.sort_values(["store_id", "item_id", "date"]).reset_index(drop=True)

long["listed"] = long["sell_price"].notna()

first_listed = (
    long[long["listed"]]
    .groupby(["store_id", "item_id"])["date"]
    .min()
    .rename("first_listed")
)
long = long.merge(first_listed, on=["store_id", "item_id"], how="left")

gaps = long[(~long["listed"]) & (long["date"] > long["first_listed"])]
print(long.shape)
print(len(gaps), "unpriced rows AFTER a series' first price")

In [ ]:
before = len(long)
long = long[long["listed"]].drop(columns=["listed", "first_listed"]).reset_index(drop=True)

print(before, "->", len(long), "rows")
print(before - len(long), "rows removed")
print(long["units"].eq(0).mean().round(3), "share of zero-sales days")
long.head()

In [ ]:
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

long.to_parquet(OUT / "sales.parquet", index=False)
calendar.to_parquet(OUT / "calendar.parquet", index=False)

check = pd.read_parquet(OUT / "sales.parquet")
print(check.shape, check.equals(long))